In [ ]:
import csv
import os
import random
import time
import uuid
from datetime import datetime, timezone

OUTPUT_PATH = "/Volumes/workspace/gagealspach/demo_source"

BATCH_INTERVAL_SEC = 5
NEW_PER_BATCH = (2, 5)       # inserts per batch, inclusive
MAX_ACTIVE = 200             # back-pressure: stop inserting past this
UPDATE_PROBABILITY = 0.25    # per-shipment chance of advancing each batch
DELETE_PROBABILITY = 0.015   # per-shipment chance of cancellation each batch

FIELDS = [
    "shipment_id", "status", "location", "customer_name",
    "_source_version", "_operation", "_source_updated_ts",
]

CUSTOMERS = ["Acme", "WidgetCo", "Globex", "Initech", "Umbrella"]

# Ordered lanes -- a shipment walks one hop per IN_TRANSIT update.
LANES = [
    ["Reno", "Sparks", "Fernley", "Sacramento"],
    ["Sacramento", "Fernley", "Sparks", "Reno"],
    ["Carson City", "Reno", "Sparks"],
    ["Sparks", "Fernley", "Carson City"],
    ["Reno", "Carson City"],
]

# In-memory state only -- nothing is read back from the lakehouse or the files.
active = {}

# Monotonic and time-seeded, so a job restart never replays a version that
# AUTO CDC has already sequenced past.
_version = int(time.time() * 1000)


def next_version():
    global _version
    _version += 1
    return _version


def new_shipment():
    lane = random.choice(LANES)
    return {
        "shipment_id": f"S{uuid.uuid4().hex[:10].upper()}",
        "status": "PENDING",
        "location": lane[0],
        "customer_name": random.choice(CUSTOMERS),
        "lane": lane,
        "hop": 0,
    }


def advance(s):
    """Move a shipment one step along its lifecycle.

    Returns False once it reaches a terminal state and should leave the
    active pool.
    """
    if s["status"] == "PENDING":
        s["status"] = "PICKED_UP"
    elif s["status"] == "PICKED_UP":
        s["status"] = "IN_TRANSIT"
        s["hop"] = 1
        s["location"] = s["lane"][1]
    elif s["status"] == "IN_TRANSIT":
        if s["hop"] < len(s["lane"]) - 1:
            s["hop"] += 1
            s["location"] = s["lane"][s["hop"]]
        if s["hop"] == len(s["lane"]) - 1:
            s["status"] = "OUT_FOR_DELIVERY"
    elif s["status"] == "OUT_FOR_DELIVERY":
        s["status"] = "DELIVERED"
        return False
    return True


def emit(s, operation, now):
    return {
        "shipment_id": s["shipment_id"],
        "status": s["status"],
        "location": s["location"],
        "customer_name": s["customer_name"],
        "_source_version": next_version(),
        "_operation": operation,
        "_source_updated_ts": now.strftime("%Y-%m-%dT%H:%M:%S"),
    }


def write_batch(rows, now):
    """Write under a hidden name, then rename, so Auto Loader never picks up
    a partially written file. Spark skips paths starting with '_'."""
    name = f"generated_{now:%Y%m%d_%H%M%S}_{uuid.uuid4().hex[:6]}.csv"
    staged = f"{OUTPUT_PATH}/_{name}"
    final = f"{OUTPUT_PATH}/{name}"

    with open(staged, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    os.rename(staged, final)

    return name


while True:
    now = datetime.now(timezone.utc)
    rows = []
    touched = set()

    # Cancellations. Anything already delivered has left the pool, so these
    # only ever hit in-flight shipments.
    for sid in list(active):
        if random.random() < DELETE_PROBABILITY:
            rows.append(emit(active.pop(sid), "DELETE", now))
            touched.add(sid)

    # Progress updates -- at most one op per shipment per batch.
    for sid in list(active):
        if sid in touched or random.random() >= UPDATE_PROBABILITY:
            continue
        s = active[sid]
        still_active = advance(s)
        rows.append(emit(s, "UPDATE", now))
        if not still_active:
            del active[sid]   # DELIVERED retires quietly; silver keeps it

    # New shipments, up to the active cap.
    room = max(MAX_ACTIVE - len(active), 0)
    for _ in range(min(random.randint(*NEW_PER_BATCH), room)):
        s = new_shipment()
        active[s["shipment_id"]] = s
        rows.append(emit(s, "INSERT", now))

    if rows:
        # CDC ordering comes from _source_version, not row order -- shuffling
        # proves the pipeline does not depend on file ordering.
        random.shuffle(rows)
        name = write_batch(rows, now)
        counts = {op: sum(1 for r in rows if r["_operation"] == op)
                  for op in ("INSERT", "UPDATE", "DELETE")}
        print(f"Wrote {name}: {counts}, {len(active)} active")

    time.sleep(BATCH_INTERVAL_SEC)